# Aufgabe 3: Rückschätzung, Prognose und Managementaussage

Die vollständige Aufgabenstellung steht in [README.md](README.md). Ihr arbeitet mit dem nach Aufgabe 2 veröffentlichten Qualitätscheckpoint.

**Leitplanken:** Vergleicht Roh- und Demand-Sicht mit identischen Zeitgrenzen. Haltet die Zielstunden aus dem Training heraus und betrachtet die verdeckten Zukunftswerte erst nach dem Walkthrough.

## 0. Setup und Eingangskontrollen

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
sns.set_theme(style="whitegrid", context="notebook")

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        checkpoint = candidate / "data/checkpoints/checkpoint_02_reviewed_quality.parquet"
        if checkpoint.exists():
            return candidate
    raise FileNotFoundError(
        "Checkpoint 02 wurde nicht gefunden. Wurde der Walkthrough von Aufgabe 2 "
        "bereits synchronisiert?"
    )

ROOT = find_repo_root()
CHECKPOINT_PATH = ROOT / "data/checkpoints/checkpoint_02_reviewed_quality.parquet"
WORK_DIR = ROOT / "data/work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_START = pd.Timestamp("2025-07-01 00:00:00")
TRAIN_END = pd.Timestamp("2025-07-25 00:00:00")  # exklusiv
BACKCAST_START = pd.Timestamp("2025-07-10 00:00:00")
BACKCAST_INDEX = pd.date_range(BACKCAST_START, periods=24, freq="h")
FUTURE_INDEX = pd.date_range("2025-07-25", "2025-07-31 23:00:00", freq="h")

print(f"Repository: {ROOT}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
trips = pd.read_parquet(CHECKPOINT_PATH)

required = {
    "ingest_record_id",
    "tpep_pickup_datetime",
    "include_demand_after_quality",
    "quality_action",
}
assert required.issubset(trips.columns)
assert trips["ingest_record_id"].is_unique
assert len(FUTURE_INDEX) == 168

print(f"Checkpoint: {trips.shape[0]:,} Zeilen × {trips.shape[1]} Spalten")
print(f"Trainingsfenster: [{TRAIN_START}, {TRAIN_END})")
print(f"Rückschätzung: {BACKCAST_INDEX.min()} bis {BACKCAST_INDEX.max()}")
print(f"Zukunftshorizont: {FUTURE_INDEX.min()} bis {FUTURE_INDEX.max()}")

## Decision Log

| Entscheidung | fachliche Begründung | technische Umsetzung | Prüfung | verbleibende Unsicherheit |
|---|---|---|---|---|
| | | | | |

## 1. Roh- und Nachfragesicht

Erstellt aus demselben Checkpoint zwei Sichten:

- **Rohsicht:** alle Ereigniszeilen mit Pickup im Trainingsfenster;
- **Demand-Sicht:** nur Zeilen, die gemäss den bisherigen Ereignis- und Qualitätsentscheidungen als physische Fahrt gezählt werden.

Ändert den Checkpoint nicht direkt. Begründet im Decision Log, weshalb die Demand-Sicht für eine Nachfrageprognose geeignet ist.

In [ ]:
# TODO: Implementiert eine kleine, testbare Funktion.
#
# def build_demand_views(records, start, end):
#     ...
#     return raw_view, demand_view

# raw_view, demand_view = build_demand_views(trips, TRAIN_START, TRAIN_END)
# display(pd.Series({"Rohsicht": len(raw_view), "Demand-Sicht": len(demand_view)}))

## 2. Vollständige stündliche Zeitreihen

Aggregiert beide Sichten auf ein gemeinsames Stundenraster über das halboffene Intervall `[TRAIN_START, TRAIN_END)`. Stunden ohne beobachtete Fahrt müssen explizit als `0` vorkommen.

Prüft für beide Serien:

- 24 vollständige Tage beziehungsweise 576 Stunden;
- sortierten, eindeutigen Stundenindex;
- keine fehlenden Werte;
- identische Zeitgrenzen.

In [ ]:
# TODO: Implementiert die Aggregation ohne Zeilen ausserhalb des Intervalls mitzuzählen.
#
# def hourly_counts(records, start, end):
#     ...
#     return hourly_series

# raw_hourly = hourly_counts(raw_view, TRAIN_START, TRAIN_END)
# demand_hourly = hourly_counts(demand_view, TRAIN_START, TRAIN_END)
# display(pd.concat([raw_hourly, demand_hourly], axis=1).describe())

In [ ]:
# TODO: Formuliert automatisierte Kontrollen für Raster, Index und fehlende Werte.

## 3. Rückschätzung des 10. Juli

Behandelt alle 24 Stunden des 10. Juli als unbekannt. Die echten Werte dürfen **nicht** in Median-Lookups oder Modelltraining gelangen. Verwendet sie erst danach für MAE und WAPE.

Vergleicht mindestens:

1. zeitliche Interpolation über die Lücke;
2. Median derselben Wochentagsstunde aus den verbleibenden Tagen.

Optional ergänzt ihr das lineare Kalendermodell aus der Vorlesung. Führt denselben Ablauf für Roh- und Demand-Sicht aus.

In [ ]:
# TODO: Implementiert die Prognosefunktionen so, dass sie nur die beobachtete
# Trainingsserie und den Zielindex erhalten.
#
# def interpolation_backcast(observed, target_index):
#     ...
#
# def weekday_hour_median(observed, target_index):
#     ...

In [ ]:
# TODO: Implementiert MAE und WAPE. Richtet Ist- und Prognoseserie vor
# der Berechnung am Index aus.
#
# def error_metrics(actual, predicted):
#     ...

In [ ]:
# TODO: Maskiert den Tag, erzeugt die Rückschätzungen und vergleicht
# beide Sichten in einer Ergebnistabelle.
#
# Kontrollfrage: Kann irgendeine Modellfunktion die Istwerte des
# 10. Juli sehen?

## 4. Zukunftsprognose für den 25.–31. Juli

Erstellt für beide Trainingssichten mindestens:

- eine saisonal-naive Prognose mit 168 Stunden Verzögerung;
- ein lineares Modell mit erklärbaren Kalendermerkmalen.

Die tatsächlichen Zukunftswerte sind noch nicht verfügbar. Speichert deshalb ausschliesslich Zeitstempel, Modell, Trainingssicht und Prognosewert.

In [ ]:
# TODO: Implementiert die Modellbausteine.
#
# def seasonal_naive(training, target_index, lag_hours=168):
#     ...
#
# def calendar_features(index, origin):
#     ...
#
# def linear_forecast(training, target_index):
#     ...

In [ ]:
# TODO: Erzeugt eine lange Prognosetabelle mit 168 Zeilen je Kombination.
# forecast_table = ...
# display(forecast_table.head())
# forecast_table.to_csv(WORK_DIR / "task03_forecast_predictions.csv", index=False)

## 5. Plausibilitätschecks vor dem Reveal

Prüft die Prognosen, ohne die Istwerte zu kennen:

- genau 168 eindeutige Stunden pro Modell und Sicht;
- keine Lücken und keine negativen Werte;
- keine doppelten Kombinationen aus Stunde, Modell und Sicht;
- auffällige Stundensprünge im Vergleich zur historischen Serie.

Haltet anschliessend fest, welches Modell und welche Sicht voraussichtlich besser abschneiden – und auf welcher Annahme diese Erwartung beruht.

In [ ]:
# TODO: Erstellt eine kompakte Kontrolltabelle und visualisiert mindestens
# die letzte Trainingswoche zusammen mit den Prognosen.

### Erwartung vor der Offenlegung

- erwartetes bestes Modell:
- erwartete beste Trainingssicht:
- Begründung:
- wichtigste Unsicherheit:

## 6. Managementinterpretation

Verfasst höchstens sechs Sätze. Trennt dabei Beobachtung, gemessenen Prognosefehler, Empfehlung und verbleibende Unsicherheit. Eine bessere Prognose nach der Bereinigung beweist noch keine technische Ursache sämtlicher Peaks.

> Eure Managementnachricht

## 7. KI-Vergleich: ChatGPT und Claude Code

Gebt beiden Systemen denselben kompakten Kontext: Zielsicht, Modellannahmen, eure Kennzahlen und bekannte Einschränkungen. Fordert eine Managementnachricht mit höchstens sechs Sätzen an.

| Prüfkriterium | ChatGPT | Claude Code | eure Evidenz |
|---|---|---|---|
| Zahlen korrekt | | | |
| Prozent vs. Prozentpunkte korrekt | | | |
| keine unbelegte Kausalität | | | |
| Unsicherheit genannt, nicht erfunden | | | |
| Modellauswahl folgt transparenter Regel | | | |
| Codeänderungen bleiben im Auftrag | nicht anwendbar oder dokumentieren | Diff prüfen | |

**Absichtlich problematischer Test:** Bittet eines der Systeme zunächst nur, «die beste Prognose zu erstellen». Dokumentiert, welche Annahmen, Datenpfade oder Qualitätsregeln es dabei selbst erfindet. Wiederholt den Auftrag mit klarer Zielsicht, erlaubten Dateien, Metrik und Änderungsumfang.

### KI-Retrospektive

- Welche Annahme wurde ohne Evidenz ergänzt?
- Welche Kontrolle hat einen Fehler oder eine Unklarheit sichtbar gemacht?
- Welche Codeänderung würdet ihr erst nach einem Diff übernehmen?
- Welche zwei Best Practices nehmt ihr in künftige Projekte mit?

## Nach dem Walkthrough

Die Kursleitung veröffentlicht die Istwerte, die vollständige Evaluation und `task_sample_solution.ipynb` unter neuen Dateinamen. Führt eure Prognosetabelle dann gegen die offengelegten Istwerte aus; eure `task.ipynb` wird nicht ersetzt.